# Línea base de detección de fraude

## 3. División temporal y preprocesamiento

Se usa el mismo archivo de los pasos 1 y 2 y se conserva la primera aparición de cada fila duplicada. El análisis exploratorio encontró un fuerte desbalance y un periodo de observación de unos dos días. Aquí ordenamos por `Time` para entrenar con el pasado y evaluar sobre transacciones posteriores.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data import load_raw, time_split, xy
from src.features import Preprocessor

df = load_raw(project_root / "data/raw/creditcard.csv")
train, val, test = time_split(df)
X_tr, y_tr = xy(train)
X_va, y_va = xy(val)
X_te, y_te = xy(test)

Los cortes apuntan al 70 % y al 85 % acumulado. Cuando varias transacciones comparten el segundo del corte, ese segundo completo pasa al conjunto siguiente. Train sirve para ajustar el modelo, validación para elegir hiperparámetros y umbral, y test para la evaluación final. El resumen cuenta los fraudes de cada conjunto.

In [2]:
splits = {"train": (train, y_tr), "val": (val, y_va), "test": (test, y_te)}
summary = pd.DataFrame.from_dict(
    {
        name: {
            "Filas": len(y),
            "Fraudes": int(y.sum()),
            "Fraude (%)": round(100 * y.mean(), 4),
            "Time mín. (s)": int(part["Time"].min()),
            "Time máx. (s)": int(part["Time"].max()),
        }
        for name, (part, y) in splits.items()
    },
    orient="index",
)

assert train["Time"].max() < val["Time"].min()
assert val["Time"].max() < test["Time"].min()
assert (summary["Fraudes"] > 0).all()
pd.testing.assert_frame_equal(pd.concat([train, val, test]), df)

summary

,Filas,Fraudes,Fraude (%),Time mín. (s),Time máx. (s)
train,198607,366,0.1843,0,132905
val,42558,55,0.1292,132906,151319
test,42561,52,0.1222,151320,172792


`Class` se conserva como etiqueta. `Time` solo determina la división. A `Amount` se le aplica `log1p` y después un `RobustScaler` ajustado con train. Se aplica el mismo escalador a los tres conjuntos y se conservan los 28 componentes PCA.

In [3]:
pre = Preprocessor().fit(X_tr)
pca_columns = [f"V{i}" for i in range(1, 29)]
train_log_amount = np.log1p(X_tr["Amount"])
q25, q75 = train_log_amount.quantile([0.25, 0.75])
center, scale = train_log_amount.median(), q75 - q25

X_tr, X_va, X_te = pre.transform(X_tr), pre.transform(X_va), pre.transform(X_te)

np.testing.assert_allclose(pre.scaler.center_, [center])
np.testing.assert_allclose(pre.scaler.scale_, [scale])
for (part, y), X in zip(splits.values(), (X_tr, X_va, X_te), strict=True):
    assert set(X.columns) == {*pca_columns, "Amount"}
    pd.testing.assert_index_equal(X.index, y.index)
    pd.testing.assert_frame_equal(X[pca_columns], part[pca_columns])
    expected_amount = (np.log1p(part["Amount"]) - center) / scale
    np.testing.assert_allclose(X["Amount"], expected_amount)

print("Verificado: tres conjuntos cronológicos con fraudes y 29 variables por conjunto.")
print("Validación y prueba usan la mediana y el rango intercuartílico de train.")

Verificado: tres conjuntos cronológicos con fraudes y 29 variables por conjunto.
Validación y prueba usan la mediana y el rango intercuartílico de train.


El conjunto de prueba contiene 52 fraudes. Un solo caso cambia el recall en aproximadamente 1.9 puntos porcentuales, por lo que el resultado final será sensible a pocos casos. Los tres pares de variables y etiquetas quedan listos para la regresión logística del paso 4.

## 4. Regresión logística balanceada

Se ajusta la regresión logística únicamente con las transacciones de entrenamiento ya preprocesadas. `class_weight="balanced"` pondera cada clase en proporción inversa a su frecuencia en train. Un puntaje más alto indica mayor riesgo de fraude.

La métrica principal es PR-AUC, calculada como *average precision*. También se reporta ROC-AUC y el mayor recall que admite cada conjunto cuando se exige al menos 80 % o 50 % de precisión. El recall a precisión fija resume la curva de cada conjunto. Para elegir un umbral operativo en los pasos siguientes se usará validación y se evaluará ese umbral fijo en prueba.

In [4]:
from sklearn.linear_model import LogisticRegression

from src.evaluate import summarize

lr = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
lr.fit(X_tr, y_tr)

evaluation_sets = [("val", X_va, y_va), ("test", X_te, y_te)]
results = pd.DataFrame([
    {"split": split, **summarize(y, lr.predict_proba(X)[:, 1], "logreg_balanced")}
    for split, X, y in evaluation_sets
]).set_index("split")
results.round(6)

,model,pr_auc,roc_auc,recall@p80,recall@p50
split,,,,,
val,logreg_balanced,0.812824,0.980591,0.745455,0.854545
test,logreg_balanced,0.689759,0.976271,0.692308,0.750000


### Referencias para interpretar el desbalance

El clasificador mayoritario predice siempre la clase más frecuente en train y muestra cuántos fraudes detecta pese a su alta exactitud. La proporción de fraude en cada conjunto es la referencia sin discriminación para PR-AUC. Se obtiene exactamente con puntajes constantes.

In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
reference_results = []
for split, X, y in evaluation_sets:
    predictions = dummy.predict(X)
    reference_results.append({
        "split": split,
        "Accuracy mayoritaria": accuracy_score(y, predictions),
        "Recall mayoritario": recall_score(y, predictions),
        "PR-AUC sin discriminación": y.mean(),
    })

pd.DataFrame(reference_results).set_index("split").round(6)

,Accuracy mayoritaria,Recall mayoritario,PR-AUC sin discriminación
split,,,
val,0.998708,0.0,0.001292
test,0.998778,0.0,0.001222


La regresión logística establece la referencia para comparar los modelos del paso 5 con las mismas variables, los mismos cortes temporales y las mismas métricas. Se usará validación para comparar y ajustar modelos. La evaluación de prueba se reserva para reportar el rendimiento.